In [4]:
import os
from pathlib import Path
import sys

# !IMPORTANT!, else Hydra isnt working !
current_path = os.getcwd()
sys.path.insert(0, Path(current_path).absolute().parents[0].as_posix())

import cv2
import tensorflow_datasets as tfds
import tensorflow as tf
import tqdm
import mediapy
import numpy as np
import torch

from uha import make_pytorch_oxe_iterable_dataset, get_octo_dataset_tensorflow, get_single_dataset_tensorflow
from torchvision.transforms.functional import convert_image_dtype
import omegaconf
from omegaconf import OmegaConf

print("All modules loaded!")

All modules loaded!


## Load Data
Next, we will load a trajectory from the Bridge dataset. We will use the one defined in uha/data/oxe/oxe_dataset_configs.py.

In [5]:
cfg = omegaconf.OmegaConf.load("../uha/data/conf/uha_default_load_config.yaml")
print(cfg)
del cfg.interleaved_dataset_cfg.frame_transform_kwargs.image_augment_kwargs # comment out to get augmented data
cfg_transforms = omegaconf.OmegaConf.load("../uha/data/conf/transforms/oxe_no_remapping.yaml")

{'_recursive_': False, 'defaults': [{'transforms': 'oxe_no_remapping'}, {'language_encoders': 'clip'}], 'DATA_NAME': 'bridge_marcel', 'DATA_PATH': '/mnt/dongxu-fs1/data-hdd/geyuan/datasets/huggingface/v1/', 'load_camera_views': ['primary', 'secondary', 'wrist'], 'action_proprio_normalization_type': 'bounds', 'interleaved_dataset_cfg': {'shuffle_buffer_size': 10000, 'balance_weights': True, 'traj_transform_kwargs': {'goal_relabeling_strategy': None, 'goal_relabeling_kwargs': {'min_bound': 20, 'max_bound': 50, 'frame_diff': 3}, 'window_size': 1, 'action_horizon': 10, 'skip_unlabeled': True}, 'frame_transform_kwargs': {'image_augment_kwargs': {'primary': {'random_resized_crop': {'scale': [0.8, 1.0], 'ratio': [0.9, 1.1]}, 'random_brightness': [0.1], 'random_contrast': [0.9, 1.1], 'random_saturation': [0.9, 1.1], 'random_hue': [0.05], 'augment_order': ['random_resized_crop', 'random_brightness', 'random_contrast', 'random_saturation', 'random_hue']}, 'secondary': {'random_resized_crop': {'s

In [6]:
print(cfg)

# dataset = get_octo_dataset_tensorflow(cfg, train=True)
dataset = get_single_dataset_tensorflow(cfg, train=True)
is_single_dataset = True
batch_size = 128
# create Pytorch Train Dataset
dataloader = make_pytorch_oxe_iterable_dataset(dataset, train=True, batch_size=batch_size, transform_dict=cfg_transforms, num_workers=0, pin_memory=True, is_single_dataset=is_single_dataset, main_process=True)
print("len in colab", len(dataloader.dataset))
it = iter(dataloader)
batch = next(it)
batch = next(it)
batch = next(it)
images, image_secondary, image_wrist = [], [], []
for i in range(batch_size):
  images.append(batch["observation"]["image_primary"][i, 0].numpy()) # [batch_size, window_size, rgb, width, height]
  image_secondary.append(batch["observation"]["image_secondary"][i, 0].numpy()) # [batch_size, window_size, rgb, width, height]
  # image_wrist.append(batch["observation"]["image_wrist"][i, 0].numpy()) # [batch_size, window_size, rgb, width, height]
  print("action", batch["action"][i, :, 0, :])

mediapy.show_video(images, fps=10)
mediapy.show_video(image_secondary, fps=10)
# mediapy.show_video(image_wrist, fps=10)

{'_recursive_': False, 'defaults': [{'transforms': 'oxe_no_remapping'}, {'language_encoders': 'clip'}], 'DATA_NAME': 'bridge_marcel', 'DATA_PATH': '/mnt/dongxu-fs1/data-hdd/geyuan/datasets/huggingface/v1/', 'load_camera_views': ['primary', 'secondary', 'wrist'], 'action_proprio_normalization_type': 'bounds', 'interleaved_dataset_cfg': {'shuffle_buffer_size': 10000, 'balance_weights': True, 'traj_transform_kwargs': {'goal_relabeling_strategy': None, 'goal_relabeling_kwargs': {'min_bound': 20, 'max_bound': 50, 'frame_diff': 3}, 'window_size': 1, 'action_horizon': 10, 'skip_unlabeled': True}, 'frame_transform_kwargs': {'resize_size': {'primary': [224, 224], 'secondary': [224, 224], 'wrist': [128, 128]}, 'resize_size_future_obs': {'primary': [112, 112], 'secondary': [112, 112], 'wrist': [128, 128]}, 'num_parallel_calls': 64}, 'traj_transform_threads': 16, 'traj_read_threads': 32}}
########################################
constructing single val dataset: cmu_stretch
########################

2025-02-22 01:47:43.092165: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


Cause: Unable to locate the source code of <function _gcd_import at 0x7f3174da1310>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f3174da1310>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


Cause: Unable to locate the source code of <function _gcd_import at 0x7f3174da1310>. Note that functions defined in certain environments, like the interactive Python shell, do not expose their source code. If that is the case, you should define them in a .py source file. If you are certain the code is graph-compatible, wrap the call using @tf.autograph.experimental.do_not_convert. Original error: could not get source code
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-02-22 01:47:44.551982: I tensorflow/core/grappler/optimizers/data/replicate_on_split.cc:32] Running replicate on split optimization


len in colab 25016


/mnt/dongxu-fs1/data-hdd/geyuan/anaconda3/envs/octo/lib/python3.9/site-packages/torch/utils/data/_utils/collate.py:171: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at ../torch/csrc/utils/tensor_numpy.cpp:206.)
  return collate([torch.as_tensor(b) for b in batch], collate_fn_map=collate_fn_map)
2025-02-22 01:47:53.488911: W tensorflow/core/kernels/data/prefetch_autotuner.cc:52] Prefetch autotuner tried to allocate 140041175040 bytes after encountering the first element of size 68379480 bytes.This already causes the autotune ram budget to be exceeded. To stay within the ram budget, either increase the ram budget or reduce element size


action tensor([[-0.1295, -1.0000, -0.6487, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1297, -1.0000, -0.6472, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1289, -1.0000, -0.6476, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1297, -1.0000, -0.6476, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1304, -1.0000, -0.6476, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1289, -1.0000, -0.6472, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1300, -1.0000, -0.6484, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1293, -1.0000, -0.6476, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1297, -1.0000, -0.6476, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1289, -1.0000, -0.6484, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1297, -1.0000, -0.6468, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1295, -1.0000, -0.6472, -1.0000, -1.0000, -1.0000,  1.0000]])
action tensor([[-0.1300, -1.

This browser does not support the video tag.
